<div dir="rtl" align="right">

# اختيارُ القنواتِ بِنسبةِ الإشارةِ إلى الضوضاءِ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَحسبُ SNR لِكلِّ قناةٍ ونَختارُ الأعلىَ جودةً.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ لِترتيبِ القنواتِ حسبَ SNR
- خريطةُ الرأسِ بِالقنواتِ المُختارةِ (أخضر) والمَرفوضة (أحمر)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| N_SELECT | 10 |
| window | 50 |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>

In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. حسابُ SNR واختيارُ القنواتِ

</div>

In [ ]:
N_SELECT = 10

def compute_snr(signal):
    signal_var = np.var(signal)
    window = 50
    moving_avg = np.convolve(signal, np.ones(window) / window, mode='same')
    noise = signal - moving_avg
    noise_var = np.var(noise)
    if noise_var == 0:
        return 0.0
    return signal_var / noise_var

snr_values = np.zeros(n_channels)
for ch in range(n_channels):
    snr_list = [compute_snr(X[trial, ch, :]) for trial in range(n_trials)]
    snr_values[ch] = np.mean(snr_list)

sorted_idx = np.argsort(snr_values)[::-1]
selected = sorted_idx[:N_SELECT]
print(f'Selected channels: {[ch_names[i] for i in selected]}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- القنواتُ المُختارةُ تَتركّزُ في المنطقةِ المركزيّةِ والجداريّة
- القنواتُ الأماميّةُ والصدغيّةُ البعيدةُ تَكونُ مَرفوضةً عادةً

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('SNR ranking', 'Topomap'))
colors = ['green' if i in selected else 'gray' for i in sorted_idx]
fig.add_trace(go.Bar(x=[ch_names[i] for i in sorted_idx], y=snr_values[sorted_idx], marker_color=colors, name='SNR'), row=1, col=1)
fig.update_xaxes(tickangle=90, row=1, col=1)

data = dataset.get_data(subjects=[1])
raw = data[1][list(data[1].keys())[0]][list(data[1][list(data[1].keys())[0]].keys())[0]]
montage = raw.get_montage()
ch_pos = montage.get_positions()['ch_pos']
ch_names_all = [ch for ch in raw.ch_names if ch in ch_pos and not ch.startswith('EOG')]
positions = np.array([ch_pos[ch] for ch in ch_names_all])
pos_2d = positions[:, :2]
scale = 1.0 / np.max(np.abs(pos_2d))
pos_2d = pos_2d * scale * 0.95

for i in range(n_channels):
    color = 'green' if i in selected else 'red'
    fig.add_trace(go.Scatter(x=[pos_2d[i, 0]], y=[pos_2d[i, 1]], mode='markers+text', text=[ch_names_all[i]], textposition='top right', marker=dict(color=color, size=10), showlegend=False), row=1, col=2)
fig.update_layout(height=500, title_text='Channel Selection by SNR')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- SNR يَقيسُ جودةَ القناةِ بِنسبةِ الإشارةِ إلى الضوضاء
- القنواتُ المُختارةُ تَتركّزُ في المنطقةِ المركزيّة
- طريقةٌ سريعةٌ لا تَتطلّبُ تدريبَ نموذج

</div>